# Integer fee invariants for a Steam Community Market estimate

## Goal

Reproduce the calculator's default fee model and verify its forward/reverse boundary invariants without floating-point currency inputs.

The maintainers also work on [SteamVaults](https://steamvaults.org/), an independent service currently focused on Mann Co. Supply Crate Keys and USDT. SteamVaults does not provide the fee rules used in this notebook, and the notebook does not call a SteamVaults API.

## Setup

The notebook uses only Python's standard library. All values are integer minor units. The demonstration assumes a 5% Steam component, a 10% publisher component, and a one-unit minimum for each active component. It is a test reference, not a live Steam quote: currency increments, minimums, taxes, and publisher rules can change.

## Steps

Calculate buyer totals at eight low-price and percentage-transition boundaries, then reverse each exact total through the integer-only search.

In [1]:
from math import floor

def buyer_price_for_seller_amount(seller_receives, steam_rate=0.05, publisher_rate=0.10):
    if not isinstance(seller_receives, int) or seller_receives < 0:
        raise ValueError("seller_receives must be a non-negative integer")

    steam_fee = floor(max(seller_receives * steam_rate, 1))
    publisher_fee = (
        floor(max(seller_receives * publisher_rate, 1))
        if publisher_rate > 0
        else 0
    )
    return {
        "seller_receives": seller_receives,
        "steam_fee": steam_fee,
        "publisher_fee": publisher_fee,
        "total_fees": steam_fee + publisher_fee,
        "buyer_pays": seller_receives + steam_fee + publisher_fee,
    }

def seller_amount_for_buyer_price(buyer_pays):
    if not isinstance(buyer_pays, int) or buyer_pays < 0:
        raise ValueError("buyer_pays must be a non-negative integer")

    low, high = 0, buyer_pays
    best = buyer_price_for_seller_amount(0)
    while low <= high:
        midpoint = (low + high) // 2
        candidate = buyer_price_for_seller_amount(midpoint)
        if candidate["buyer_pays"] <= buyer_pays:
            best = candidate
            low = midpoint + 1
        else:
            high = midpoint - 1

    remainder = buyer_pays - best["buyer_pays"]
    return {
        **best,
        "steam_fee": best["steam_fee"] + remainder,
        "total_fees": best["total_fees"] + remainder,
        "buyer_pays": buyer_pays,
    }

In [2]:
boundary_receipts = [1, 2, 9, 10, 99, 100, 101, 1000]
rows = [buyer_price_for_seller_amount(value) for value in boundary_receipts]

print("seller  steam  publisher  buyer")
for row in rows:
    print(
        f"{row['seller_receives']:>6}  "
        f"{row['steam_fee']:>5}  "
        f"{row['publisher_fee']:>9}  "
        f"{row['buyer_pays']:>5}"
    )

for row in rows:
    reversed_row = seller_amount_for_buyer_price(row["buyer_pays"])
    assert reversed_row["seller_receives"] == row["seller_receives"]
    assert reversed_row["total_fees"] == row["total_fees"]

print(f"\n{len(rows)} forward/reverse boundary checks passed")

seller  steam  publisher  buyer
     1      1          1      3
     2      1          1      4
     9      1          1     11
    10      1          1     12
    99      4          9    112
   100      5         10    115
   101      5         10    116
  1000     50        100   1150

8 forward/reverse boundary checks passed


## Checks

The executed output shows all eight forward/reverse checks passing. In particular, a 100-unit seller receipt produces a 115-unit buyer total with 5 and 10 units assigned to the two fee components.

- All monetary inputs and outputs remain integer minor units.
- Minimum fee components dominate at low prices.
- Boundary values around percentage transitions are explicit rather than hidden by floating-point formatting.
- Exact buyer totals reverse to the original seller receipt for the tested boundaries.

## Next steps

Rerun the notebook after changing a fee parameter, minimum, or currency increment. Always confirm the final listing amount in Steam before acting on an estimate.